In [1]:
"""
Assignment 1: Reinforcement Learning from Human Feedback (RLHF)
Notebook: 03_reward_helpsteer2.ipynb

Purpose:
This notebook trains a reward model using the NVIDIA HelpSteer2 dataset.
Because HelpSteer2 does not directly provide pairwise chosen/rejected responses,
the notebook constructs preference pairs by selecting stronger and weaker responses
based on helpfulness-related scores, then trains a reward model on those pairs.

Summary of sections:
1. Environment setup and package imports
   - Imports required libraries and prepares the reward-model training environment.

2. Reward model and tokenizer loading
   - Loads the base model / classification architecture used for reward modelling.

3. Dataset loading
   - Loads the HelpSteer2 dataset with prompt, response, and quality-related scores.

4. Preference pair construction
   - Converts HelpSteer2 records into pairwise preference samples by selecting
     higher-quality responses as chosen and lower-quality responses as rejected.

5. Dataset preprocessing
   - Formats the constructed preference pairs into the structure required for training.

6. Reward training configuration
   - Defines training arguments, evaluation setup, and other model settings.

7. Reward model training
   - Trains the reward model to predict relative response quality from the generated pairs.

8. Validation / evaluation
   - Measures the quality of the reward model on held-out preference data.

9. Saving outputs
   - Saves the trained HelpSteer2 reward model and associated files.

Main output:
- Reward model trained on HelpSteer2-derived preference pairs, later used in PPO training.
"""

'\nAssignment 1: Reinforcement Learning from Human Feedback (RLHF)\nNotebook: 03_reward_helpsteer2.ipynb\n\nPurpose:\nThis notebook trains a reward model using the NVIDIA HelpSteer2 dataset.\nBecause HelpSteer2 does not directly provide pairwise chosen/rejected responses,\nthe notebook constructs preference pairs by selecting stronger and weaker responses\nbased on helpfulness-related scores, then trains a reward model on those pairs.\n\nSummary of sections:\n1. Environment setup and package imports\n   - Imports required libraries and prepares the reward-model training environment.\n\n2. Reward model and tokenizer loading\n   - Loads the base model / classification architecture used for reward modelling.\n\n3. Dataset loading\n   - Loads the HelpSteer2 dataset with prompt, response, and quality-related scores.\n\n4. Preference pair construction\n   - Converts HelpSteer2 records into pairwise preference samples by selecting\n     higher-quality responses as chosen and lower-quality res

In [6]:
# Install packages

!pip -q install transformers datasets trl accelerate scikit-learn


# Import Libraries

import torch
import numpy as np
from collections import defaultdict
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl import RewardTrainer, RewardConfig


# Model and Configuration

MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "./outputs/rm_helpsteer2"

SEED = 42
TRAIN_SAMPLES = 1000
EVAL_SAMPLES = 100
MAX_LENGTH = 512


# Load tokenizer and model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.sep_token

if tokenizer.eos_token is None:
    tokenizer.eos_token = tokenizer.sep_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

print("Reward model and tokenizer loaded successfully.")
print("Model name:", MODEL_NAME)
print("PAD token:", tokenizer.pad_token)
print("PAD token id:", tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token)
print("EOS token id:", tokenizer.eos_token_id)


# Load HelpSteer2 dataset

raw = load_dataset("nvidia/HelpSteer2")

print("Available splits:", list(raw.keys()))
for split_name in raw.keys():
    print(split_name, raw[split_name].column_names)
    print(raw[split_name][0])
    break


# Helper functions

def render_text(prompt, response):
    prompt = "" if prompt is None else str(prompt)
    response = "" if response is None else str(response)

    if prompt.strip():
        return f"Human: {prompt}\n\nAssistant: {response}"
    return response

def build_preference_dataset_from_scores(split_ds):
    """
    Build chosen/rejected pairs by grouping rows with the same prompt.
    For each prompt, sort candidate responses by helpfulness.
    Pair highest-helpfulness response against lowest-helpfulness response.
    Keep only groups with at least 2 responses and different helpfulness scores.
    """

    groups = defaultdict(list)

    for ex in split_ds:
        prompt = ex.get("prompt", "")
        response = ex.get("response", "")
        helpfulness = ex.get("helpfulness", None)

        if prompt is None or response is None or helpfulness is None:
            continue

        groups[str(prompt)].append({
            "response": str(response),
            "helpfulness": float(helpfulness)
        })

    rows = []
    skipped = 0

    for prompt, candidates in groups.items():
        if len(candidates) < 2:
            skipped += 1
            continue

        # sort by helpfulness descending
        candidates = sorted(candidates, key=lambda x: x["helpfulness"], reverse=True)

        best = candidates[0]
        worst = candidates[-1]

        # skip ties
        if best["helpfulness"] == worst["helpfulness"]:
            skipped += 1
            continue

        chosen = render_text(prompt, best["response"])
        rejected = render_text(prompt, worst["response"])

        if chosen == rejected:
            skipped += 1
            continue

        rows.append({
            "chosen": chosen,
            "rejected": rejected,
            "chosen_helpfulness": best["helpfulness"],
            "rejected_helpfulness": worst["helpfulness"]
        })

    print(f"Built {len(rows)} preference rows; skipped {skipped}.")
    return Dataset.from_list(rows)


# Build train/eval preference datasets

train_source = raw["train"]
eval_source = raw["validation"] if "validation" in raw else raw["train"]

train_pref = build_preference_dataset_from_scores(train_source)
eval_pref = build_preference_dataset_from_scores(eval_source)

train_pref = train_pref.shuffle(seed=SEED)
eval_pref = eval_pref.shuffle(seed=SEED)

train_pref = train_pref.select(range(min(TRAIN_SAMPLES, len(train_pref))))
eval_pref = eval_pref.select(range(min(EVAL_SAMPLES, len(eval_pref))))

print("Train preference size:", len(train_pref))
print("Eval preference size:", len(eval_pref))

if len(train_pref) == 0 or len(eval_pref) == 0:
    raise ValueError("No preference pairs were built from HelpSteer2. Check dataset schema.")

print(train_pref[0])


# Training configuration

reward_config = RewardConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    bf16=False,
    fp16=False,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    max_length=MAX_LENGTH,
    gradient_checkpointing=False,
)


# Create reward trainer

trainer = RewardTrainer(
    model=model,
    args=reward_config,
    train_dataset=train_pref,
    eval_dataset=eval_pref,
    processing_class=tokenizer,
)

print("RewardTrainer created successfully.")


# Train reward model

trainer.train()


# Save reward model

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Reward model saved to:", OUTPUT_DIR)


# Evaluate on validation set using pairwise accuracy

eval_subset = eval_pref.select(range(len(eval_pref)))

chosen_scores = []
rejected_scores = []

trained_model = trainer.model
trained_model.eval()

for item in eval_subset:
    chosen_text = item["chosen"]
    rejected_text = item["rejected"]

    chosen_inputs = tokenizer(
        chosen_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(trained_model.device)

    rejected_inputs = tokenizer(
        rejected_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(trained_model.device)

    with torch.no_grad():
        chosen_score = trained_model(**chosen_inputs).logits.squeeze().item()
        rejected_score = trained_model(**rejected_inputs).logits.squeeze().item()

    chosen_scores.append(chosen_score)
    rejected_scores.append(rejected_score)

chosen_scores = np.array(chosen_scores)
rejected_scores = np.array(rejected_scores)

pairwise_accuracy = np.mean(chosen_scores > rejected_scores)

print("=" * 80)
print("Validation pairwise accuracy:", pairwise_accuracy)
print("Average chosen score:", chosen_scores.mean())
print("Average rejected score:", rejected_scores.mean())


# Show a few sample comparisons

raw_eval = eval_pref.select(range(min(3, len(eval_pref))))

for i, sample in enumerate(raw_eval):
    chosen_text = sample["chosen"]
    rejected_text = sample["rejected"]

    chosen_enc = tokenizer(
        chosen_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(trained_model.device)

    rejected_enc = tokenizer(
        rejected_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(trained_model.device)

    with torch.no_grad():
        chosen_score = trained_model(**chosen_enc).logits.squeeze().item()
        rejected_score = trained_model(**rejected_enc).logits.squeeze().item()

    print("=" * 100)
    print(f"Example {i+1}")
    print(f"Chosen helpfulness   : {sample['chosen_helpfulness']}")
    print(f"Rejected helpfulness : {sample['rejected_helpfulness']}")
    print(f"Chosen score         : {chosen_score:.4f}")
    print(f"Rejected score       : {rejected_score:.4f}")
    print("Chosen preview :")
    print(chosen_text[:500])
    print("\nRejected preview :")
    print(rejected_text[:500])


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Reward model and tokenizer loaded successfully.
Model name: distilbert-base-uncased
PAD token: [PAD]
PAD token id: 0
EOS token: [SEP]
EOS token id: 102
Available splits: ['train', 'validation']
train ['prompt', 'response', 'helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity']
{'prompt': 'c#', 'response': 'C# is a high-level, object-oriented programming language developed by Microsoft as part of its .NET initiative. It was created as a modern alternative to Java and supports a variety of programming paradigms, including imperative, functional, and event-driven. C# is primarily used for Windows application development, but it can also be used for web, mobile, and game development. The language is designed to be safe, secure, and efficient, and it provides developers with a rich set of libraries and tools for building robust and scalable applications. C# is also widely used in the game development industry, particularly in the development of games for the Xbox 360 and Xbox

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Reward model saved to: ./outputs/rm_helpsteer2
Validation pairwise accuracy: 0.61
Average chosen score: 0.05937012697802856
Average rejected score: 0.05531684788526036
Example 1
Chosen helpfulness   : 4.0
Rejected helpfulness : 2.0
Chosen score         : 0.0456
Rejected score       : 0.0416
Chosen preview :
Human: give me some of the literature survey of phishing websites using data mining

Assistant: Phishing is a type of cyber attack that involves creating fake websites to steal personal information, such as login credentials and credit card numbers. Data mining is a technique used to analyze large amounts of data to identify patterns and trends. It can be used to detect phishing websites by analyzing various features, such as URL, content, and user behavior.

Here are some recent literature surv

Rejected preview :
Human: give me some of the literature survey of phishing websites using data mining

Assistant: Sure, here are some examples of literature surveys on phishing websites us

In [7]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_root = "/content/drive/MyDrive/rlhf_assignment_outputs"
os.makedirs(drive_root, exist_ok=True)

target_path = os.path.join(drive_root, "rm_helpsteer2")

if os.path.exists(target_path):
    shutil.rmtree(target_path)

shutil.copytree(OUTPUT_DIR, target_path)

print("Copied RM-HelpSteer2 to Drive:", target_path)
print("Files in Drive copy:")
for f in os.listdir(target_path):
    print(" ", f)

Mounted at /content/drive
Copied RM-HelpSteer2 to Drive: /content/drive/MyDrive/rlhf_assignment_outputs/rm_helpsteer2
Files in Drive copy:
  tokenizer.json
  config.json
  model.safetensors
  tokenizer_config.json
  checkpoint-32
  training_args.bin
  README.md
  checkpoint-4
